In [0]:
# =========================================================
# 03_silver
# Camada SILVER — Data Quality
# =========================================================

# ---------------------------------------------------------
# 1. Criar schema SILVER
# ---------------------------------------------------------

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")


# ---------------------------------------------------------
# 2. Imports
# ---------------------------------------------------------

from pyspark.sql.functions import col, upper


# =========================================================
# 3. CLIENTES
# =========================================================

# Ler BRONZE
df_clientes = spark.read.table("bronze.clientes")

# Remover duplicados
df_clientes = df_clientes.dropDuplicates()

# Remover nulos importantes
df_clientes = df_clientes.filter(
    col("id_cliente").isNotNull()
)

# Padronizar nomes em maiúsculo
df_clientes = df_clientes.withColumn(
    "nome",
    upper(col("nome"))
)

# Preencher valores nulos
df_clientes = df_clientes.na.fill({
    "cidade": "NÃO INFORMADO",
    "estado": "NÃO INFORMADO"
})

# Salvar SILVER
df_clientes.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.clientes")


# =========================================================
# 4. PRODUTOS
# =========================================================

# Ler BRONZE
df_produtos = spark.read.table("bronze.produtos")

# Remover duplicados
df_produtos = df_produtos.dropDuplicates()

# Remover produtos sem ID
df_produtos = df_produtos.filter(
    col("id_produto").isNotNull()
)

# Remover preços inválidos
df_produtos = df_produtos.filter(
    col("preco") > 0
)

# Padronizar nomes
df_produtos = df_produtos.withColumn(
    "nome",
    upper(col("nome"))
)

# Preencher categoria nula
df_produtos = df_produtos.na.fill({
    "categoria": "NÃO INFORMADO"
})

# Salvar SILVER
df_produtos.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.produtos")


# =========================================================
# 5. VENDAS
# =========================================================

# Ler BRONZE
df_vendas = spark.read.table("bronze.vendas")

# Remover duplicados
df_vendas = df_vendas.dropDuplicates()

# Remover IDs nulos
df_vendas = df_vendas.filter(
    col("id_venda").isNotNull()
)

# Quantidade deve ser maior que zero
df_vendas = df_vendas.filter(
    col("quantidade") > 0
)

# Valor unitário deve ser maior que zero
df_vendas = df_vendas.filter(
    col("valor_unitario") > 0
)

# Salvar SILVER
df_vendas.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.vendas")


# =========================================================
# 6. Validação
# =========================================================

print("✅ Tabelas SILVER criadas com sucesso!")

print("\n📋 Tabelas disponíveis no schema SILVER:")
spark.sql("SHOW TABLES IN silver").show(truncate=False)